In [ ]:
# load dataset filenames from data\train_filenames.lst and data\test_filenames.lst
train_filenames_path = "data/train_filenames.lst"
with open(train_filenames_path, "r") as f:
    train_filenames = f.read().splitlines()
test_filenames_path = "data/test_filenames.lst"
with open(test_filenames_path, "r") as f:
    test_filenames = f.read().splitlines()

print(f"Number of train images: {len(train_filenames)}")
print(f"Number of test images: {len(test_filenames)}")
print(f"Total number of images: {len(train_filenames) + len(test_filenames)}")


In [ ]:
import rasterio
import numpy as np

# 200 m tif images from csv file
images_200m = []

dataset_directory_200m = "C:\\Users\\uceda\\Documents\\uni\\master\\ENLIGHT\\course\\data\\dataset\\s2\\200m"

max_h, max_w = 21, 21

# Pad to max size
images_200m = []
for filename in train_filenames:
    with rasterio.open(dataset_directory_200m + "\\" + filename) as src:
        image = src.read()  # (12, H, W)
    pad_h = max_h - image.shape[1]
    pad_w = max_w - image.shape[2]
    # Pad with zeros on the right/bottom to make all images the same size (12, max_h, max_w)
    padded = np.pad(image, ((0, 0), (0, pad_h), (0, pad_w)), mode='constant')
    images_200m.append(padded)

images_200m = np.array(images_200m)  # shape: (N, 12, max_H, max_W)

# save to csv file
np.savetxt("images_200m.csv", images_200m.reshape(images_200m.shape[0], -1), delimiter=",")

In [ ]:
import rasterio
import random
import matplotlib.pyplot as plt
import numpy as np

def scale(band, pmin=2, pmax=98):   # to remove outliers
    lo, hi = np.percentile(band, (pmin, pmax))
    return np.clip((band - lo) / (hi - lo), 0, 1)

img = random.choice(images_200m)

band_names = ['Band 1 - Coastal aerosol', 'Band 2 - Blue', 'Band 3 - Green', 'Band 4 - Red', 'Band 5 - Vegetation red edge', 'Band 6 - Vegetation red edge', 'Band 7 - Vegetation red edge', 'Band 8 - NIR', 'Band 8A - Narrow NIR', 'Band 9 - Water vapour', 'Band 10 - SWIR - Cirrus', 'Band 11 - SWIR', 'Band 12 - SWIR']
# plot each of the 12 bands
with rasterio.open(img, 'r') as ds:
  fig, axs = plt.subplots(3, 4, figsize=(20, 15))
  print('img shape:', ds.shape)
  print('img dtype:', ds.dtypes)
  print('img count (number of bands):', ds.count)
  print('img descriptions:', ds.descriptions)
  for i in range(12):
    band = ds.read(i+1).astype(np.float32)
    axs[i//4, i%4].imshow(scale(band), cmap='viridis')
    axs[i//4, i%4].set_title(f"{band_names[i]}")
    axs[i//4, i%4].axis("off")
  plt.tight_layout()
  plt.show()

In [ ]:
species_list = [
    'Abies_alba', 
    'Acer_pseudoplatanus',
    'Alnus_spec', 
    'Betula_spec', 
    'Cleared', 
    'Fagus_sylvatica', 
    'Fraxinus_excelsior', 
    'Larix_decidua', 
    'Larix_kaempferi', 
    'Picea_abies', 
    'Pinus_nigra', 
    'Pinus_strobus', 
    'Pinus_sylvestris', 
    'Populus_spec', 
    'Prunus_spec', 
    'Pseudotsuga_menziesii', 
    'Quercus_petraea', 
    'Quercus_robur', 
    'Quercus_rubra', 
    'Tilia_spec'
]

In [ ]:
import pandas as pd
loaded_arr = np.loadtxt("images_200m.csv", delimiter=",")
b_names = [f"band_{i}" for i in range(1, 13)]
df = pd.DataFrame(loaded_arr, columns=[f"{b}_{i}" for b in b_names for i in range(1, max_h*max_w+1)])

one_hot_rows = []

for idx, filename in enumerate(train_filenames):
    species = None
    species_idx = None
    for j, s in enumerate(species_list):
        if s in filename:
            species = s
            species_idx = j
            break

    df.at[idx, 'species'] = species

    # one-hot vector: all zeros if no match, else 1 at species_idx
    one_hot_label = [1 if k == species_idx else 0 for k in range(len(species_list))]
    one_hot_rows.append(one_hot_label)

# create separate df with one vector column per sample (no individual species columns)
one_hot_df = pd.DataFrame({
    'one_hot': [row for row in one_hot_rows]
})

one_hot_df.to_csv('labels_one_hot.csv', index=False)
df.to_csv('labels_df.csv', index=False)

print(f"df shape: {df.shape}")
print(f"one_hot_df shape: {one_hot_df.shape}")
print(f"Vector length: {len(one_hot_rows[0])} (covers all {len(species_list)} species)")
one_hot_df.head()
